## Notebook 07 — arreglo de variables de entorno + generador de informe final

Objetivo de hoy:
1. Confirmar que el fix de GROQ_MODEL_* no rompió nada.
2. Probar el pipeline completo (con Revisor y Adaptador reales) contra
   UN solo documento, para no gastar cuota de golpe.
3. Si sale limpio, construir report_generator.py para ver el docx final.

No se redefine ningún agente aquí — todos se importan desde src_agents/,
ya con el fix aplicado en analyst.py, redactor_v1.py y adaptador_en.py.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
from src_agents.agents.analyst import agente_analista
from src_agents.agents.redactor_v1 import agente_redactor
from src_agents.agents.adaptador_en import agente_adaptador_en

print("Los tres importan bien con las variables nuevas")

### Paso 2: pipeline completo contra un solo documento

Usamos la carpeta "prueba" data/raw/prueba,
con un único Excel. Así, si algo falla, sabemos que no es un problema de volumen — y si Groq corta la cuota, se corta rápido, sin haber gastado ya media cuota diaria en los otros dos documentos.

In [ ]:
DATA_DIR_TEST = PROJECT_ROOT / "data" / "raw" / "prueba"
assert DATA_DIR_TEST.exists(), f"No encuentro la carpeta en {DATA_DIR_TEST}"
print("Archivos en la carpeta de prueba:", list(DATA_DIR_TEST.iterdir()))

### Ejecutar el pipeline completo

Ingesta -> Analista -> Redactor -> Revisor -> Adaptador, con el grafo real que ya está fusionado en dev. Si Groq corta por límite, queremos ver hasta dónde llegó, no perder toda la información con un traceback.

In [ ]:
from src_agents.graph.workflow import pipeline
from groq import APIStatusError

try:
    resultado = pipeline.invoke({"uploaded_files": [str(DATA_DIR_TEST)]})
    print("Pipeline completo, sin errores")
    print(f"Revisión: {resultado['review']}")
    print(f"Draft ES: {len(resultado['draft'])} caracteres")
    print(f"Draft EN: {len(resultado.get('draft_en', ''))} caracteres")
except APIStatusError as e:
    print(f"Se cortó a mitad de camino: {e}")

## Por qué "Empleo" no mostró este problema y "Hoja1" sí

Empleo es una tabla de indicadores numéricos — cada ConceptoValor es
una cifra corta y verificable letra a letra.

Hoja1 (agencia_innovacion...) es un informe de plan estratégico —
columnas como "PRINCIPALES AVANCES" o "LINEAS DE ACTUACIÓN" contienen
descripciones narrativas completas, no datos atómicos. El Analista
las trató igual que cualquier otro concepto->valor (correctamente,
según su diseño), pero el Revisor no está preparado para validar
texto largo con comparación literal.

### Celda — usa el draft ya capturado, sin volver a llamar a Groq

Guardamos el resultado de la ejecución anterior en una variable, en vez de volver a invocar el pipeline. Esto nos permite construir el generador de DOCX sin depender de que la cuota de qwen se resetee.

In [ ]:
# Pega aquí el draft que ya viste impreso arriba, como texto fijo,
# solo para esta prueba de hoy — no es la forma final de obtenerlo
draft_es_prueba = resultado["draft"]  # si "resultado" sigue en memoria del kernel
# Si has reiniciado el kernel y ya no está en memoria, dímelo y lo recreamos
# a partir de lo que pegaste en el chat.

### Notebook 07 — Paso 3: generador de informe (report_generator.py)

Objetivo: convertir state["draft"] (y state["draft_en"] cuando lo
haya) en un documento .docx real, con las incidencias del Revisor
como sección de notas para revisión humana — no las escondemos.

Probamos primero en el notebook con el draft ya capturado en memoria
(sin gastar cuota), y solo cuando funcione lo graduamos a
src_agents/services/report_generator.py (archivo ya existe vacío
en el scaffold).

In [ ]:
from docx import Document
from docx.shared import Pt
from pathlib import Path


def generar_informe_docx(estado: dict, ruta_salida: Path) -> Path:
    """Genera un .docx a partir del estado del pipeline: borrador en
    español, traducción al inglés si existe, y las incidencias del
    Revisor como notas de validación al final (no se ocultan — sigue
    habiendo supervisión humana antes de publicar el informe)."""
    doc = Document()

    doc.add_heading("Memoria Anual de Actividades", level=0)
    doc.add_heading("Ayuntamiento de San Sebastián de los Reyes", level=2)

    doc.add_heading("Informe (Español)", level=1)
    for parrafo in estado["draft"].split("\n\n"):
        if parrafo.strip():
            doc.add_paragraph(parrafo.strip())

    draft_en = estado.get("draft_en", "")
    if draft_en:
        doc.add_page_break()
        doc.add_heading("Report (English)", level=1)
        for parrafo in draft_en.split("\n\n"):
            if parrafo.strip():
                doc.add_paragraph(parrafo.strip())

    review = estado.get("review")
    if review is not None and review.incidencias:
        doc.add_page_break()
        doc.add_heading("Notas de validación (revisión humana pendiente)", level=1)
        p = doc.add_paragraph(
            "El Agente Revisor detectó las siguientes cifras o datos del "
            "Analista que no aparecen tal cual en el texto redactado. "
            "Revisar antes de publicar:"
        )
        p.runs[0].italic = True
        for incidencia in review.incidencias:
            doc.add_paragraph(incidencia, style="List Bullet")

    doc.save(ruta_salida)
    return ruta_salida


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

ruta = generar_informe_docx(resultado, OUTPUT_DIR / "informe_prueba.docx")
print(f"Generado en: {ruta}")

In [ ]:
print(resultado["draft_en"])

## Notebook 07 — Prueba mínima: reasoning_effort="none" en el Adaptador (issue #90)

Antes de probar el pipeline completo, se valida el cambio aislado: se llama
directamente a `agente_adaptador_en` con un borrador corto inventado (sin pasar
por Ingesta/Analista/Redactor), para no gastar cuota de más.

Con `reasoning_effort="none"` añadido en `ChatGroq(...)`, el modelo
(qwen/qwen3.6-27b) no debería generar ningún bloque `<think>` — ni en el
intento normal ni en el método de respaldo. Si el resultado sale limpio aquí,
se da por validado el cambio sin necesidad de correr el pipeline entero.

In [ ]:
from src_agents.agents.adaptador_en import agente_adaptador_en

estado_prueba = {"draft": "Durante 2025 se registraron 78 personas en la Red, superando la meta de 10."}
resultado = agente_adaptador_en(estado_prueba)
print(resultado["draft_en"])
print("---")
print("Contiene <think>:", "<think>" in resultado["draft_en"])